## 📚 Cell 1 · Base
Downloads or updates the code from GitHub, connects Google Drive (approve the popup), creates the project folders and `api_keys.toml`. Run first, and again whenever you want the latest code.

In [ ]:
# Downloads (or updates) the code from GitHub, then connects Google Drive: approve the popup.
REPO = "https://github.com/adriexnet/yt-channel-studio.git"
VERSION = "main"          # a branch or a release tag, e.g. "v2.0.0", to stay on a fixed version

import os, sys, shutil, subprocess, importlib
CODE = "/content/yt-channel-studio"
DRIVE_COPY = "/content/temp_drive/MyDrive/YT-Channel-Studio/code"   # last working copy, kept in your Drive
git = lambda *args: subprocess.run(["git", *args], capture_output=True, text=True)

if os.path.isdir(f"{CODE}/.git"):
    result = git("-C", CODE, "fetch", "-q", "--tags", "--force", "origin")
    if result.returncode == 0:
        is_branch = git("-C", CODE, "ls-remote", "--heads", "origin", VERSION).stdout.strip()
        result = git("-C", CODE, "checkout", "-q", "--force", "--detach", f"origin/{VERSION}" if is_branch else VERSION)
else:
    result = git("clone", "-q", REPO, CODE)
    if result.returncode == 0:
        result = git("-C", CODE, "checkout", "-q", "--detach", VERSION)
if result.returncode != 0:
    if not os.path.isdir(CODE):
        from google.colab import drive
        drive.mount("/content/temp_drive")
        if not os.path.isdir(DRIVE_COPY):
            raise RuntimeError("Could not download the code from GitHub and there is no copy in Drive yet:\n" + result.stderr)
        shutil.copytree(DRIVE_COPY, CODE)
    print("⚠️  GitHub did not answer; using the copy already available:", result.stderr.strip()[-200:])

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "toml"], check=True)
sys.dont_write_bytecode = True   # no cached .pyc: an update made in the same second is never skipped
sys.path.insert(0, CODE)
import studio
importlib.reload(studio)
studio.reload()           # picks up the new code without restarting the runtime
print("Code version:", studio.__version__, git("-C", CODE, "log", "-1", "--format=%h %cd", "--date=short").stdout.strip())

from studio import setup
setup.base()

## 📦 Cell 2 · Environment
System tools, cloudflared, uv, the notebook packages and the subtitle fonts. Without a GPU it tells you what is unavailable instead of stopping.

In [ ]:
from studio import setup
setup.environment()

## 🧠 Cell 3 · Ollama *(optional)*
Local LLMs (weights persist in Drive) + chat UI. Skip it if you use an external LLM.

In [ ]:
from studio.tools import ollama
ollama.start(models=["qwen2.5:14b", "deepseek-r1:14b"])

## 🎨 Cell 4 · Forge WebUI *(optional)*
FLUX images in a Python 3.10 venv: local covers, base images and AI photos.

In [ ]:
from studio.tools import forge
forge.start()   # models: flux_dev, flux_fill, flux_vae, clip_l, t5xxl, qwen_edit (all by default)

## 🎬 Cell 5 · ComfyUI *(optional)*
Local AI video (Wan 2.2 / LTX-Video) in a Python 3.11 venv.

In [ ]:
from studio.tools import comfyui
comfyui.start(models=["wan22", "ltxv"])

## 🎙️ Cell 6 · Audio Studio *(optional)*
Try any voice before using it in a reel. `f5=False` skips the F5-TTS voice-cloning install.

In [ ]:
from studio.tools import audio
audio.start(f5=True)

## 🚀 Cell 7 · Control Center
Idea → script → images → voice → clips → music → montage → publishing, with live progress, API keys and your saved reels.

In [ ]:
from studio import control_center
control_center.start()

## 📋 Cell 8 · Dashboard
Every running tool and its public link.

In [ ]:
from studio import dashboard
dashboard.show()